In [ ]:
# Run from the repository root, or open this notebook anywhere inside it.
from pathlib import Path
import os
_repo_candidates = [Path.cwd(), *Path.cwd().parents]
_project_root = next((p for p in _repo_candidates if (p / "code/data_process").is_dir() and (p / "README.md").is_file()), None)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.chdir(_project_root)


In [ ]:
import pandas as pd
import os

# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "matched_df.csv")
df = pd.read_csv(file_path)

In [ ]:
print(len(df))

In [ ]:
df.columns

# Standardize values


In [ ]:
# List of variables to standardize [degree type related]
dg_vars = ['D2DG', 'D3DG', 'D4DG', 'D5DG']

# Before conversion
print("Before conversion:")
for var in dg_vars:
    print(f"\n{var} unique values:", df[var].unique())

# Convert float values to strings without decimal points
for var in dg_vars:
    # Convert non-'L' values to integers (this will handle both '1.0' and '1' to 1)
    mask = (df[var] != 'L') & df[var].notna()
    df.loc[mask, var] = df.loc[mask, var].astype(float).astype(int).astype(str)
    
# After conversion
print("\nAfter conversion:")
for var in dg_vars:
    print(f"\n{var} value counts:")
    print(df[var].value_counts(dropna=False))

In [ ]:
print(df['DGRDG'].unique())

In [ ]:
df.columns

# Filtering

In [ ]:
# 1. valid cohort 'BAYR'
df = df[df['BAYR']< 9998.0]

print(f'# Obs valid cohort:{len(df)}')

# 2. necessary salary range

df = df[df['SALARY'] > 0]

print(f'# Obs positive salary:{len(df)}')


In [ ]:
df['SALARY'].describe()

# merge UR data

In [ ]:
# Define the path and read the CSV files
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")
ur_ma_path = os.path.join(base_path, "ur_ma_16y.csv")

# Read the files
ur_ba_16y = pd.read_csv(ur_ba_path)
ur_ma_16y = pd.read_csv(ur_ma_path)
# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})
ur_ma_16y = ur_ma_16y.rename(columns={'CGMD16O': 'UR'})

In [ ]:
ur_ma_16y

In [ ]:
# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year
ur_ba_16y_yearly = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
ur_ba_16y_yearly = ur_ba_16y_yearly.rename(columns={'year': 'BAYR', 'UR': 'gradur_ba'})

# For Master's
ur_ma_16y['observation_date'] = pd.to_datetime(ur_ma_16y['observation_date'])
ur_ma_16y['year'] = ur_ma_16y['observation_date'].dt.year
ur_ma_16y_yearly = ur_ma_16y.groupby('year')['UR'].mean().reset_index()
ur_ma_16y_yearly = ur_ma_16y_yearly.rename(columns={'year': 'BAYR', 'UR': 'gradur_ma'})

# Create separate dataframes for bachelor's and master's degree holders
df_ba = df[df['DGRDG'] == 1].copy()
df_ma = df[df['DGRDG'] == 2].copy()

# Merge unemployment rates
df_ba = df_ba.merge(ur_ba_16y_yearly, on='BAYR', how='left')
df_ma = df_ma.merge(ur_ma_16y_yearly, on='BAYR', how='left')

# Combine back and create final gradur column
df_ba['gradur'] = df_ba['gradur_ba']
df_ma['gradur'] = df_ma['gradur_ma']
df_final = pd.concat([df_ba, df_ma])

# Check the results
print("Bachelor's degrees:")
print(df_final[df_final['DGRDG']==1][['BAYR', 'gradur']].sort_values('BAYR').drop_duplicates().to_string(index=False))

print("\nMaster's degrees:")
print(df_final[df_final['DGRDG']==2][['BAYR', 'gradur']].sort_values('BAYR').drop_duplicates().to_string(index=False))

# Check sample sizes
print("\nSample sizes:")
print(df_final.groupby(['BAYR', 'DGRDG'])['gradur'].agg(['count', 'mean']).round(2))

In [ ]:
# Check for missing values before preparing regression variables
print("Missing values in original data:")
print(df_final[['SALARY', 'BAYR', 'REFYR', 'DGRDG', 'gradur','assigned_id']].isnull().sum())

# Clean the data
df_final = df_final.dropna(subset=['SALARY', 'BAYR', 'REFYR', 'DGRDG', 'gradur'])

# drop obs without assigned id (multiple matches within year)
df_final = df_final.dropna(subset=['assigned_id'])


In [ ]:
len(df_final)

In [ ]:
df_final.to_csv('data-all-waves/df_final.csv', index = False)
